# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu-yadav-10/Flyrank-ML-starter-template/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Unit of analysis + time window

**One row = one `report_date` × client × content item** — a single page's measured search (GSC) and analytics (GA4) performance for one day. My lane reads that river at one mid-panel month, **`month=2026-03`**, and aggregates it to the unit I actually score: **one row per page** (331,437 pages were alive in March).

The five contract answers, in plain words:

1. **What one row means:** for my lane, one *page* — built by summing that page's daily rows across March.
2. **Tables I use:** `fact_content_daily_performance/month=2026-03` for behavior, joined to `dim_content` (staleness metadata). `dim_clients` is read for history context only. The April partition is touched **only** to measure the label window — never features.
3. **Time window:** features cover the fully closed window **2026-03-01 → 2026-03-31**; the decision moment is **2026-04-01**; the label is measured strictly after it (April vs March impressions). Feature window and label window never overlap.
4. **What I predict / rank:** each page's probability that April impressions fall below 80% of March ("visibility decline next month"), used to order a refresh-review queue.
5. **Deliberately excluded:** `provider_used` / `model_used` — which LLM wrote a page is FlyRank's internal product decision, not an observable market signal; learning it would model our own ops history. (Also excluded by construction: anything measured after April 1.)

Why iterate on a mid-panel month: the `_sample` table is exactly the final month (June 2026), so developing label logic there would mean developing inside my own sealed test window.

In [2]:
import os, getpass

os.environ.setdefault('HF_HUB_DISABLE_PROGRESS_BARS', '1')
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
FACT_MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)"
FACT_APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)"
print('warehouse reachable - dim_clients rows:', con.sql(f'SELECT COUNT(*) FROM {DIM_CLIENTS}').fetchone()[0])

warehouse reachable - dim_clients rows: 104


## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket. The rule that sorts them: **could I know this value on 2026-04-01, for a decision about April?** If yes → feature (if it isn't an ID). If it *is* the April measurement itself, or computed from it → label bucket, never a feature.

- **Context** (grouping, joining, splitting, reading — never learned from): `client_hash_id`, `content_hash_id`, `report_date`, `month`; lifecycle flags (`is_published`, `is_deleted`) used only to filter.
- **Features** (all aggregated over the closed March window, plus dim_content metadata that exists before the decision moment): impressions, clicks, avg position, GA4 sessions (only days where `ga4_data_available` IS TRUE), content age/staleness, keyword context.
- **Label / proxy:** `imp_apr` (April impression total) and anything computed with it — including the demo trap column below.
- **Excluded:** `provider_used`, `model_used` (internal product-decision flags); `url_hash_id`, `keyword_hash_id` (identifiers — join keys only); all post-April-1 information, especially the June `_sample` table.

Missing values follow patterns, not chance: GA4 columns are zero-filled or NULL outside each client's GA4 coverage (the three-valued flag), and dim_content metadata is blank along content-type lines — so availability flags, not blind fills, decide what counts as data.

In [4]:
import pandas as pd

contract = pd.DataFrame([
    ('client_hash_id', 'context', 'groups pages by client; also the split key'),
    ('content_hash_id', 'context', 'the page identity; queue keys'),
    ('report_date, month', 'context', 'defines the windows'),
    ('gsc_impressions, gsc_clicks, gsc_avg_position', 'feature', 'March sums/means over closed window'),
    ('ga4_sessions (IS TRUE days only)', 'feature', 'measured audience in March'),
    ('content_updated_date -> days_since_update', 'feature', 'staleness known before Apr 1'),
    ('imp_apr (April impressions)', 'label/proxy', 'the outcome window; never a feature'),
    ('anything derived from imp_apr', 'label/proxy', 'incl. the trap column demonstrated below'),
    ('provider_used, model_used', 'excluded', 'internal product flags, not market signals'),
    ('url_hash_id, keyword_hash_id', 'excluded', 'identifiers: joins only, never learned'),
], columns=['field(s)', 'bucket', 'why'])
display(contract)

prod_flags = con.sql(f"""
    SELECT COUNT(*)                    AS total_pages,
           COUNT(provider_used)        AS with_provider_flag,
           COUNT(DISTINCT provider_used) AS distinct_flags
    FROM {DIM_CONTENT}
""").df()
print(prod_flags.to_string(index=False))
print('-> a handful of fixed flags written by our own platform pipeline:')
print('   product-decision metadata about how pages get made, not market signal — excluded as promised.')

,field(s),bucket,why
0,client_hash_id,context,groups pages by client; also the split key
1,content_hash_id,context,the page identity; queue keys
2,"report_date, month",context,defines the windows
3,"gsc_impressions, gsc_clicks, gsc_avg_position",feature,March sums/means over closed window
4,ga4_sessions (IS TRUE days only),feature,measured audience in March
5,content_updated_date -> days_since_update,feature,staleness known before Apr 1
6,imp_apr (April impressions),label/proxy,the outcome window; never a feature
7,anything derived from imp_apr,label/proxy,incl. the trap column demonstrated below
8,"provider_used, model_used",excluded,"internal product flags, not market signals"
9,"url_hash_id, keyword_hash_id",excluded,"identifiers: joins only, never learned"


 total_pages  with_provider_flag  distinct_flags
      519606              149670               9
-> a handful of fixed flags written by our own platform pipeline:
   product-decision metadata about how pages get made, not market signal — excluded as promised.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three proofs, all on the mid-panel month `month=2026-03`. A contract claim without a query next to it is a guess.

**Q1 — Grain:** if one row truly is one report_date × client × content page-day, grouping by those three columns must return zero violators.

**Q2 — Count + span:** how many page-days live in my slice, and do they really cover March 1–31?

**Q3 — Availability:** how many rows survive a `ga4_data_available IS TRUE` filter? (The flag is three-valued — TRUE / FALSE / NULL — so plain boolean filtering silently mishandles NULLs; only `IS TRUE` counts real measurements.)

In [5]:
q1 = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MAR}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print('Q1 grain violators (expect 0 rows):', len(q1))
if len(q1):
    display(q1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Q1 grain violators (expect 0 rows): 0


In [6]:
q2 = con.sql(f"""
    SELECT COUNT(*) AS page_days, MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM {FACT_MAR}
""").df()
print(q2.to_string(index=False))

 page_days  first_day   last_day
   9841378 2026-03-01 2026-03-31


In [7]:
q3 = con.sql(f"""
    SELECT COUNT(*) AS page_days,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true,
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null_unmeasured
    FROM {FACT_MAR}
""").df()
print(q3.to_string(index=False))
row = q3.iloc[0]
print(f"survivors: GSC {row['gsc_true'] / row['page_days']:.1%} | GA4 {row['ga4_true'] / row['page_days']:.1%} of page-days")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 page_days  gsc_true  ga4_true  ga4_null_unmeasured
   9841378   3611061    413966              3018741
survivors: GSC 36.7% | GA4 4.2% of page-days


### Five features, no more

Aggregating the March partition to one row per page (joined to `dim_content`). Every feature passes the same test — knowable at the decision moment (2026-04-01):

| # | feature | what it says | knowable at the decision moment because… |
|---|---|---|---|
| 1 | `imp_log` | ln(1 + March impressions) — demand scale | the March window fully closed on Mar 31 |
| 2 | `ctr_pct` | 100 × clicks ÷ impressions in March — snippet appeal | both inputs are March-only sums |
| 3 | `pos_avg` | mean GSC position over ranked days (unranked → 100) — visibility quality | positions are observed daily up to Mar 31 |
| 4 | `sessions_log` | ln(1 + GA4 sessions, summed **only over `ga4_data_available IS TRUE` days**) — real-world audience | Q3 showed only ~4% of page-days carry true GA4 data; the filter keeps artifacts out |
| 5 | `days_since_update` | Apr 1 − `content_updated_date` — staleness | update dates live in dim_content, written before April |

Modeling subset: pages with ≥ 100 March impressions (stable denominators — same floor the guided notebook uses); label `is_declining_next` = April impressions < 80% of March, with pages absent from April counted as collapsed (0 impressions).

In [8]:
feats_sql = f"""
    WITH mar AS (
        SELECT client_hash_id, content_hash_id,
               SUM(COALESCE(gsc_impressions, 0)) AS imp_mar,
               SUM(COALESCE(gsc_clicks, 0))      AS clicks_mar,
               AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_raw,
               SUM(CASE WHEN ga4_data_available IS TRUE THEN COALESCE(ga4_sessions, 0) ELSE 0 END) AS sess_true
        FROM {FACT_MAR}
        GROUP BY 1, 2
    )
    SELECT c.client_hash_id,
           c.content_hash_id,
           ln(1 + m.imp_mar)                                                 AS imp_log,
           CASE WHEN m.imp_mar > 0 THEN 100.0 * m.clicks_mar / m.imp_mar END AS ctr_pct,
           COALESCE(m.pos_raw, 100.0)                                        AS pos_avg,
           ln(1 + m.sess_true)                                               AS sessions_log,
           date_diff('day', c.content_updated_date, DATE '2026-04-01')       AS days_since_update,
           m.imp_mar
    FROM mar m
    JOIN {DIM_CONTENT} c USING (client_hash_id, content_hash_id)
"""
features = con.sql(feats_sql).df()
print('pages alive in March:', f'{len(features):,}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages alive in March: 331,437


In [9]:
label_sql = f"""
    SELECT m.client_hash_id, m.content_hash_id,
           m.imp_mar,
           COALESCE(a.imp_apr, 0) AS imp_apr
    FROM (
        SELECT client_hash_id, content_hash_id, SUM(COALESCE(gsc_impressions, 0)) AS imp_mar
        FROM {FACT_MAR} GROUP BY 1, 2
    ) m
    LEFT JOIN (
        SELECT client_hash_id, content_hash_id, SUM(COALESCE(gsc_impressions, 0)) AS imp_apr
        FROM {FACT_APR} GROUP BY 1, 2
    ) a USING (client_hash_id, content_hash_id)
    WHERE m.imp_mar >= 100
"""
labels = con.sql(label_sql).df()

data = (features.drop(columns='imp_mar')
                .merge(labels, on=['client_hash_id', 'content_hash_id'], validate='one_to_one'))
data['is_declining_next'] = (data['imp_apr'] < 0.8 * data['imp_mar']).astype(int)
data['days_since_update'] = data['days_since_update'].fillna(data['days_since_update'].median())

FEATURE_COLS = ['imp_log', 'ctr_pct', 'pos_avg', 'sessions_log', 'days_since_update']
print('modeling rows (>= 100 March impressions):', f'{len(data):,}')
print('decline-next-month base rate:', round(data['is_declining_next'].mean(), 3))
print('remaining NaNs in features:', int(data[FEATURE_COLS].isna().sum().sum()))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

modeling rows (>= 100 March impressions): 101,441
decline-next-month base rate: 0.517
remaining NaNs in features: 0


### The trap, performed on purpose

My five honest features are weak-ish by design — they should land around AUC ≈ 0.59 against next-month decline. Now the mistake everyone makes: add ONE column computed from the outcome window — `trap_ratio = imp_apr / max(imp_mar, 1)` — which is literally the label's own ingredients. Watch the score sprint toward perfect. Then delete it and confirm the honest number is the real one.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(split.split(data, groups=data['client_hash_id']))

def quick_auc(feature_cols):
    clf = RandomForestClassifier(n_estimators=60, random_state=42, n_jobs=-1)
    clf.fit(data.loc[tr_idx, feature_cols], data['is_declining_next'].iloc[tr_idx])
    proba = clf.predict_proba(data.loc[te_idx, feature_cols])[:, 1]
    return roc_auc_score(data['is_declining_next'].iloc[te_idx], proba)

auc_honest = quick_auc(FEATURE_COLS)
print(f'HONEST   AUC, five March-only features:      {auc_honest:.3f}')

data['trap_ratio'] = data['imp_apr'] / data['imp_mar'].clip(lower=1)
auc_trapped = quick_auc(FEATURE_COLS + ['trap_ratio'])
print(f'TRAPPED  AUC, outcome-derived ratio added:   {auc_trapped:.3f}')

data = data.drop(columns='trap_ratio')
auc_after_delete = quick_auc(FEATURE_COLS)
print(f'DELETED  AUC after removing the trap column: {auc_after_delete:.3f}')
print()
print(f'the jump ({auc_honest:.3f} -> {auc_trapped:.3f}) was pure leakage: trap_ratio is computed')
print('from the same April measurement as the label, so the model was just re-reading the answer.')
print(f'kept number: {auc_after_delete:.3f} — directional signal, useful only as decision-support ranking.')

HONEST   AUC, five March-only features:      0.593
TRAPPED  AUC, outcome-derived ratio added:   1.000
DELETED  AUC after removing the trap column: 0.593

the jump (0.593 -> 1.000) was pure leakage: trap_ratio is computed
from the same April measurement as the label, so the model was just re-reading the answer.
kept number: 0.593 — directional signal, useful only as decision-support ranking.


## 4. Data limits

**Named limitation — the unbalanced panel.** Clients join the platform at different times, so equal calendar months are not equal amounts of history. The check below shows 43 of 104 clients started GSC on/after 2025-10-01 (and 37 have no recorded start at all): for them, early-2026 months record a *ramp-up*, so a March→April drop can be measurement timing rather than decay. Any past→future label I build must respect each client's own history start, not one global calendar.

Secondary limits I keep in view: only ~4% of March page-days carry `ga4_data_available IS TRUE`, so `sessions_log` measures a thin slice of reality (~31% more are NULL-unmeasured, invisible to any filter); pages that vanish from April are labeled full declines (548 here) even when the reason could be deletion; and `fact_content_query_90d` overlaps recent months, so its columns are unusable as features for labels living near the panel's end. Net effect on claims: everything above is **observed association on one month-pair of one release — directional decision-support, not causal proof about search engines**.

In [11]:
history = con.sql(f"""
    SELECT COUNT(*) AS clients,
           COUNT(*) FILTER (WHERE gsc_data_start >= DATE '2025-10-01') AS started_gsc_oct25_or_later,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL)              AS gsc_start_null
    FROM {DIM_CLIENTS}
""").df()
print(history.to_string(index=False))

 clients  started_gsc_oct25_or_later  gsc_start_null
     104                          43              37


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.